### Estimating Transfer Function - Square Wave
- took multiple vibrometer measurements of square wave played from speaker, using various distances from the sensor head to the speaker

In [ ]:
#pip install sounddevice

In [ ]:
import numpy as np
import scipy.signal as signal
import sounddevice as sd
import matplotlib.pyplot as plt
import pandas as pd
from math import gcd
import scipy.fft
from scipy.signal import welch
from scipy.signal import butter, sosfilt

#### Generate Square Wave:
- sample rate: 44100 samples/sec
- duration: 5 seconds
- frequency: 100 Hz

In [ ]:
# Generate square wave
fs = 44100  # Sample rate
duration = 5  # seconds
t = np.linspace(0, duration, int(fs * duration))
freq = 100  # Hz
square_wave = signal.square(2 * np.pi * freq * t)

# Play and record
#recording = sd.playrec(square_wave, fs, channels=1)
#sd.wait()

# Estimate transfer function (Method 2)
# f, Pxy = signal.csd(square_wave, recording[:, 0], fs)
# f, Pxx = signal.welch(square_wave, fs)
# H = Pxy / Pxx  # Transfer function estimate

# Plot input signal:
plt.figure(figsize=(12,2))
plt.plot(t[:1000], square_wave[:1000], 'b-', linewidth=1)
#plt.plot(t, square_wave, 'b-', linewidth=1)
#plt.xlim(right=1)
plt.xlabel('Time (s)')
plt.ylabel('Amplitude') # units?
plt.title('Input Signal (Square Wave)')
plt.grid(True, alpha=0.3)
plt.show()

#### Measurements:
- Played square wave at a constant volume from bluetooth speaker for meausrements
- Used vibrometer to take measurements
- Took multiple measurements with the speaker at different distances from the vibrometer sensor head

##### Measurement Summary:
- Measurement 1 (m1): speaker about 70cm from sensor head
- Measurement 2 (m2): speaker about 80cm from sensor head
- Measurement 3 (m3): speaker about 90cm from sensor head
- Measurement 4 (m4): speaker about 100cm from sensor head
- Measurement 5 (m5): speaker about 110cm from sensor head
- Measurement 6 (m6): speaker about 120cm from sensor head

In [ ]:
# Load vibrometer measurements:
column_names1= ['Time1 (s)', 'Signal1 (m)']
m1 = pd.read_table("square-wave-test1.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names1)
#m1.head()

column_names2=['Time2 (s)', 'Signal2 (m)']
m2 = pd.read_table("square-wave-test2.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names2)
#m2.head()

column_names3=['Time3 (s)', 'Signal3 (m)']
m3 = pd.read_table("square-wave-test3.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names3)
#m3.head()

column_names4=['Time4 (s)', 'Signal4 (m)']
m4 = pd.read_table("square-wave-test4.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names4)
#m4.head()

column_names5=['Time5 (s)', 'Signal5 (m)']
m5 = pd.read_table("square-wave-test5.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names5)
#m5.head()

column_names6=['Time6 (s)', 'Signal6 (m)']
m6 = pd.read_table("square-wave-test6.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names6)
#m6.head()

#### Plot Measurements:

In [ ]:
# Plot measurements:
measurements= [m1, m2, m3, m4, m5, m6]
for i, dframe in enumerate(measurements):
    time_col = f"Time{i+1} (s)"
    signal_col = f"Signal{i+1} (m)"
    # trim time
    dframe = dframe[dframe[time_col] >= 1.5]
    dframe = dframe[dframe[time_col] <= 7.5]
    # Plot:
    plt.figure(figsize=(12,4))
    plt.plot(dframe[time_col], dframe[signal_col], label=f"Output Signal - Measurement {i+1}")
    plt.grid(True, alpha=0.4)
    plt.title(f"Output Signal - Measurement {i+1}")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (m)")
    #plt.xlim(1.8,2.5)
    plt.show()

#### Resample Measurements:
- Resampling measurements to match sample rate of input signal (fs = 44100)
- Using signal.resample_poly to resample the measurements 

In [ ]:
# Resample measurements to match sample rate of input signal (fs=44100)
# Try signal.resample.poly:
for i, dframe in enumerate(measurements):
    time_col = f"Time{i+1} (s)"
    signal_col = f"Signal{i+1} (m)"
    #type(resampled_measurement) --> numpy array
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Check the sample rate of the original data
    original_duration = dframe[time_col].iloc[-1] - dframe[time_col].iloc[0]
    original_num_samples = len(dframe[signal_col])
    measurement_sr = original_num_samples / original_duration
    print(f"Original duration: {original_duration} seconds")
    print(f"Original number of samples: {original_num_samples}")
    print(f"Measurement sample rate: {measurement_sr} Hz")
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Calculate the ratio
    up = 44100
    down = 12500
    g = gcd(up, down)
    up_simplified = up // g      # 441
    down_simplified = down // g  # 125
    print(f"Resampling ratio: {up_simplified}/{down_simplified}")
    
    resampled_measurement = signal.resample_poly(dframe[signal_col], 441, 125)
    
    # Create correct time array
    dt = 1/44100
    resampled_time = np.arange(len(resampled_measurement)) * dt + dframe[time_col].iloc[0]
    
    # Verify:
    print(f"Original duration: {dframe[time_col].iloc[-1] - dframe[time_col].iloc[0]:.2f} s")
    print(f"Resampled duration: {resampled_time[-1] - resampled_time[0]:.2f} s")
    print(f"New sample rate: {1/(resampled_time[1]-resampled_time[0]):.1f} Hz")
    print("resampled_time[-1]:", resampled_time[-1])
    # Plot:
    plt.figure(figsize=(12,4))
    plt.plot(resampled_time, resampled_measurement, label=f"Resampled Output Measurement {i+1}")
    plt.grid(True, alpha=0.4)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (m)")
    #plt.xlim(4,4.2)
    plt.title(f"Resampled Output - Measurement{i+1}")
    plt.show()

In [ ]:
resampled_m1 = signal.resample_poly(m1["Signal1 (m)"], 441, 125)
resampled_m2 = signal.resample_poly(m2["Signal2 (m)"], 441, 125)
resampled_m3 = signal.resample_poly(m3["Signal3 (m)"], 441, 125)
resampled_m4 = signal.resample_poly(m4["Signal4 (m)"], 441, 125)
resampled_m5 = signal.resample_poly(m5["Signal5 (m)"], 441, 125)
resampled_m6 = signal.resample_poly(m6["Signal6 (m)"], 441, 125)

resampled_measurements = [resampled_m1, resampled_m2, resampled_m3, resampled_m4, resampled_m5, resampled_m6]

dt = 1/44100
resampled_t1 = np.arange(len(resampled_m1)) * dt + m1["Time1 (s)"].iloc[0]
resampled_t2 = np.arange(len(resampled_m2)) * dt + m2["Time2 (s)"].iloc[0]
resampled_t3 = np.arange(len(resampled_m3)) * dt + m3["Time3 (s)"].iloc[0]
resampled_t4 = np.arange(len(resampled_m4)) * dt + m4["Time4 (s)"].iloc[0]
resampled_t5 = np.arange(len(resampled_m5)) * dt + m5["Time5 (s)"].iloc[0]
resampled_t6 = np.arange(len(resampled_m6)) * dt + m6["Time6 (s)"].iloc[0]

resampled_times = [resampled_t1, resampled_t2, resampled_t2, resampled_t3, resampled_t4, resampled_t5, resampled_t6]

# Put resampled data into new dataframes:
resampled_m1 = pd.DataFrame({'Resampled Measurement1 (m)': resampled_m1, 'Resampled Time1 (s)': resampled_t1})
resampled_m2 = pd.DataFrame({'Resampled Measurement2 (m)': resampled_m2, 'Resampled Time2 (s)': resampled_t2})
resampled_m3 = pd.DataFrame({'Resampled Measurement3 (m)': resampled_m3, 'Resampled Time3 (s)': resampled_t3})
resampled_m4 = pd.DataFrame({'Resampled Measurement4 (m)': resampled_m4, 'Resampled Time4 (s)': resampled_t4})
resampled_m5 = pd.DataFrame({'Resampled Measurement5 (m)': resampled_m5, 'Resampled Time5 (s)': resampled_t5})
resampled_m6 = pd.DataFrame({'Resampled Measurement6 (m)': resampled_m6, 'Resampled Time6 (s)': resampled_t6})
resampled_measurements = [resampled_m1, resampled_m2, resampled_m3, resampled_m4, resampled_m5, resampled_m6]

##### Remove Signal Offset from Shifting Laser
- Trying to correct for vertical shift from laser moving around on surface of speaker
- Adjust threshold as needed for detecting start/end times of square wave

In [ ]:
# trying to correct for vertical shift from laser moving around on surface of speaker:
def remove_dc_offset_segmented(signal, time, auto_detect=True):
    """Remove DC offset by detecting and correcting each segment"""
    if auto_detect:
        # Detect signal start/stop using amplitude threshold
        abs_signal = np.abs(signal - np.median(signal[:100]))
        threshold = 0.25e-5
    
        active = abs_signal > threshold
        
        # Find transitions
        diff = np.diff(active.astype(int))
        start_idx = np.where(diff == 1)[0]
        end_idx = np.where(diff == -1)[0]
        
        if len(start_idx) > 0 and len(end_idx) > 0:
            start = start_idx[0]
            end = end_idx[-1]
        else:
            # assume middle is active
            start = int(0.15 * len(signal))
            end = int(0.85 * len(signal))
    
    # Calculate mean of active region only
    active_mean = np.mean(signal[start:end])
    
    # Create corrected signal
    corrected_signal = signal.copy()
    corrected_signal[start:end] -= active_mean
    
    return corrected_signal, start, end

corrected_m1, start_m1, end_m1 = remove_dc_offset_segmented(resampled_m1["Resampled Measurement1 (m)"], resampled_m1["Resampled Time1 (s)"])
corrected_m2, start_m2, end_m2 = remove_dc_offset_segmented(resampled_m2["Resampled Measurement2 (m)"], resampled_m2["Resampled Time2 (s)"])
corrected_m3, start_m3, end_m3 = remove_dc_offset_segmented(resampled_m3["Resampled Measurement3 (m)"], resampled_m3["Resampled Time3 (s)"])
corrected_m4, start_m4, end_m4 = remove_dc_offset_segmented(resampled_m4["Resampled Measurement4 (m)"], resampled_m4["Resampled Time4 (s)"])
corrected_m5, start_m5, end_m5 = remove_dc_offset_segmented(resampled_m5["Resampled Measurement5 (m)"], resampled_m5["Resampled Time5 (s)"])
corrected_m6, start_m6, end_m6 = remove_dc_offset_segmented(resampled_m6["Resampled Measurement6 (m)"], resampled_m6["Resampled Time6 (s)"])

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def plot_comparison(time, original, corrected, start=None, end=None):
    """Plot original vs corrected signal"""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
    ax1.plot(time, original)
    ax1.set_ylabel('Amplitude (m)')
    ax1.set_title('Original Signal')
    ax1.set_xlim(1.5,8)
    ax1.grid(True, alpha=0.3)
    if start and end:
        ax1.axvspan(time[start], time[end], alpha=0.2, color='red', label='Active region')
        ax1.legend()
    ax2.plot(time, corrected)
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('Amplitude (m)')
    ax2.set_title('Corrected Signal (Zero-Mean)')
    ax2.set_xlim(1.5, 8)
    ax2.grid(True, alpha=0.3)
    ax2.axhline(y=0, color='k', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

print("Measurement 1:")
plot_comparison(resampled_m1["Resampled Time1 (s)"], resampled_m1["Resampled Measurement1 (m)"], corrected_m1, start_m1, end_m1)
print("Measurement 2:")
plot_comparison(resampled_m2["Resampled Time2 (s)"], resampled_m2["Resampled Measurement2 (m)"], corrected_m2, start_m2, end_m2)
print("Measurement 3:")
plot_comparison(resampled_m3["Resampled Time3 (s)"], resampled_m3["Resampled Measurement3 (m)"], corrected_m3, start_m3, end_m3)
print("Measurement 4:")
plot_comparison(resampled_m4["Resampled Time4 (s)"], resampled_m4["Resampled Measurement4 (m)"], corrected_m4, start_m4, end_m4)
print("Measurement 5:")
plot_comparison(resampled_m5["Resampled Time5 (s)"], resampled_m5["Resampled Measurement5 (m)"], corrected_m5, start_m5, end_m5)
print("Measurement 6:")
plot_comparison(resampled_m6["Resampled Time6 (s)"], resampled_m6["Resampled Measurement6 (m)"], corrected_m6, start_m6, end_m6)

##### Trim Measurements to 'Active" Region:
- 'Active' region: time range where square wave is detected, based on remove_dc_offset_segmented function

In [ ]:
def trim_to_active_region(time, signal, start, end, padding=0.0):
    """Trim the signal to only include the active region
    Parameters:
    - time: time from dataframe
    - signal: signal from dataframe 
    - start: start index of active region
    - end: end index of active region
    - padding: additional time (in seconds) to include before/after active region
    Returns:
    - time_trimmed: trimmed time array
    - signal_trimmed: trimmed signal array
    - new_start: new start index in trimmed array (will be 0 if no padding)
    - new_end: new end index in trimmed array"""
    time= time.values
    signal = signal.values
    if padding > 0:
        # Calculate padding in samples
        dt = np.mean(np.diff(time))
        padding_samples = int(padding / dt)
        
        # Expand the region with padding
        start_with_padding = max(0, start - padding_samples)
        end_with_padding = min(len(signal), end + padding_samples)
    else:
        start_with_padding = start
        end_with_padding = end
    
    # Trim arrays
    time_trimmed = time[start_with_padding:end_with_padding]
    signal_trimmed = signal[start_with_padding:end_with_padding]
    
    # Calculate new indices relative to trimmed array
    new_start = start - start_with_padding
    new_end = end - start_with_padding
    
    # Reset time to start at 0
    time_trimmed = time_trimmed - time_trimmed[0]
    
    return time_trimmed, signal_trimmed, new_start, new_end
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
time_trimmed1, signal_trimmed1, new_start1, new_end1 = trim_to_active_region(resampled_m1["Resampled Time1 (s)"], corrected_m1, start_m1, end_m1)
time_trimmed2, signal_trimmed2, new_start2, new_end2 = trim_to_active_region(resampled_m2["Resampled Time2 (s)"], corrected_m2, start_m2, end_m2)
time_trimmed3, signal_trimmed3, new_start3, new_end3 = trim_to_active_region(resampled_m3["Resampled Time3 (s)"], corrected_m3, start_m3, end_m3)
time_trimmed4, signal_trimmed4, new_start4, new_end4 = trim_to_active_region(resampled_m4["Resampled Time4 (s)"], corrected_m4, start_m4, end_m4)
time_trimmed5, signal_trimmed5, new_start5, new_end5 = trim_to_active_region(resampled_m5["Resampled Time5 (s)"], corrected_m5, start_m5, end_m5)
time_trimmed6, signal_trimmed6, new_start6, new_end6 = trim_to_active_region(resampled_m6["Resampled Time6 (s)"], corrected_m6, start_m6, end_m6)

trimmed_signals = [signal_trimmed1, signal_trimmed2, signal_trimmed3, signal_trimmed4, signal_trimmed5, signal_trimmed6]
trimmed_times = [time_trimmed1, time_trimmed2, time_trimmed3, time_trimmed4, time_trimmed5, time_trimmed6]
# Plot: ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
for i, (time, signal) in enumerate(zip(trimmed_times, trimmed_signals)):
    plt.figure(figsize=(12, 4))
    plt.plot(time, signal)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (m)")
    #plt.xlim(right=1)
    plt.grid(True)
    plt.title(f"Trimmed Measurement {i+1}")
    plt.show()

In [ ]:
# Make new dataframes with the trimmed signals:
M1_data = {'Trimmed Time1 (s)': time_trimmed1, 'Trimmed Signal1 (m)': signal_trimmed1}
M1 = pd.DataFrame(M1_data)
#M1.head()
M2_data = {'Trimmed Time2 (s)': time_trimmed2, 'Trimmed Signal2 (m)': signal_trimmed2}
M2 = pd.DataFrame(M2_data)
#M2.head()
M3_data = {'Trimmed Time3 (s)': time_trimmed3, 'Trimmed Signal3 (m)': signal_trimmed3}
M3 = pd.DataFrame(M3_data)
#M3.head()
M4_data = {'Trimmed Time4 (s)': time_trimmed4, 'Trimmed Signal4 (m)': signal_trimmed4}
M4 = pd.DataFrame(M4_data)
#M4.head()
M5_data = {'Trimmed Time5 (s)': time_trimmed5, 'Trimmed Signal5 (m)': signal_trimmed5}
M5 = pd.DataFrame(M5_data)
#M5.head()
M6_data = {'Trimmed Time6 (s)': time_trimmed6, 'Trimmed Signal6 (m)': signal_trimmed6}
M6 = pd.DataFrame(M6_data)
#M6.head()
trimmed_measurements = [M1, M2, M3, M4, M5, M6]

##### Detrend and Shift Resampled Measurements to Zero Mean:

In [ ]:
# Detrend and Shift Resampled Measurements to Zero Mean:
for i, dframe in enumerate(trimmed_measurements):
    measurement_col = f"Trimmed Signal{i+1} (m)"
    time_col = f"Trimmed Time{i+1} (s)"
    detrended_measurement_col = f"Detrended Measurement{i+1} (m)"
    mean_shifted_col = f"Detrended/Shifted Measurement {i+1} (m)"
   # Detrend
    dframe[detrended_measurement_col] = scipy.signal.detrend(dframe[measurement_col], type='linear') 
    # Then Shift Mean to ~Zero (without normalizing)
    dframe[mean_shifted_col] = dframe[detrended_measurement_col] - np.mean(dframe[detrended_measurement_col])
    print(f"M{i+1} Mean:", np.mean(dframe[mean_shifted_col]))
    # Plot:
    plt.figure(figsize=(12,4))
    plt.plot(dframe[time_col], dframe[mean_shifted_col], label=f"Measurement {i+1}")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (m)")
    plt.grid(True,alpha=0.4)
    plt.axhline(y=0.0, label="0.0", color='red', linestyle=":")
    plt.title(f"Measurement {i+1} - Detrended and Shifted to Zero Mean")
    #plt.xlim(2,2.2)
    plt.tight_layout()
    plt.show()

#### Align Measurements in Time:
- Note: Time alignment between input and output measurements is important for accurately estimating the phase component of the transfer function
- Method 1: Find the start of the square wave based on a threshold value
    - Function returns the index where the signal amplitude first exceeds a threshold value
    - Can use the index value to shift the measurements in time to match a reference measurement

In [ ]:
# Method 1: Find start of square wave based on threshold amplitude value
def find_square_wave_start(signal, threshold=1.0e-5):
    """Find the index where the signal first exceeds threshold
    signal: the measurement array
    threshold: absolute amplitude threshold"""
    # Find first sample where absolute value exceeds threshold
    start_idx = np.argmax(np.abs(signal) > threshold)
    
    # Verify if actually found a crossing (argmax returns 0 if nothing found)
    if np.abs(signal[start_idx]) <= threshold:
        print(f"Warning: No sample found above threshold {threshold}")
        return 0
    
    return start_idx
    
# Plot:
for i, dframe in enumerate(trimmed_measurements):
    signal_col = f"Detrended/Shifted Measurement {i+1} (m)"
    time_col = f"Trimmed Time{i+1} (s)"
    start_idx = find_square_wave_start(dframe[signal_col])
    print("start index:", start_idx)
    plt.figure(figsize=(12,4))
    plt.plot(dframe[time_col], dframe[signal_col], label=f'Measurement {i+1}')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude (m)')
    plt.title(f"Square Wave Start Detection - Measurement {i+1}")
    plt.grid(True, alpha=0.4)
    plt.axvline(x=dframe[time_col].iloc[start_idx], color='red', linestyle='--', label='detected start')
    #plt.xlim(2.0,2.5)
    plt.legend()
    plt.show()

##### Reference Measurement for Horizontally Shifting Measurements: M3
- M3 has the earliest index for the start of the square wave compared to other measurements --> start index: 12
- Now can subtract M3 start index from each of the other start indecies to get value to shift each measurement by:
    - M1: 5142 -  12 = 5130
    - M2: 20 - 12 = 8
    - M4: 15 - 12 = 3
    - M5: 40 - 12 = 28
    - M6: 2343 - 12 = 2331

In [ ]:
# Horizontally shift measurements based on find_square_wave_start results:
# Reference measurement: M3 (because wave starts at earliest index compared to other measurements)
#print("M3 start time:", M3["Trimmed Time3 (s)"].iloc[12])
t3_ref = resampled_m3["Resampled Time3 (s)"].iloc[12] # reference start time

# Shift times based on reference start time (t3_ref):
t1_hshifted = M1["Trimmed Time1 (s)"] - (M1["Trimmed Time1 (s)"].iloc[5142] - t3_ref)
t2_hshifted = M2["Trimmed Time2 (s)"] - (M2["Trimmed Time2 (s)"].iloc[20] - t3_ref)
t4_hshifted = M4["Trimmed Time4 (s)"] - (M4["Trimmed Time4 (s)"].iloc[15] - t3_ref)
t5_hshifted = M5["Trimmed Time5 (s)"] - (M5["Trimmed Time5 (s)"].iloc[40] - t3_ref)
t6_hshifted = M6["Trimmed Time6 (s)"] - (M6["Trimmed Time6 (s)"].iloc[2343] - t3_ref)

plt.figure(figsize=(12,4))
plt.plot(M3["Trimmed Time3 (s)"], M3["Detrended/Shifted Measurement 3 (m)"], label='M3 (reference)', alpha=0.5)
plt.plot(t1_hshifted, M1["Detrended/Shifted Measurement 1 (m)"], label='shifted M1', alpha=0.5)
plt.plot(t2_hshifted, M2["Detrended/Shifted Measurement 2 (m)"], label='shifted M2', alpha=0.5)
plt.plot(t4_hshifted, M4["Detrended/Shifted Measurement 4 (m)"], label='shifted M4', alpha=0.5)
plt.plot(t5_hshifted, M5["Detrended/Shifted Measurement 5 (m)"], label='shifted M5', alpha=0.5)
plt.plot(t6_hshifted, M6["Detrended/Shifted Measurement 6 (m)"], label='shifted M6', alpha=0.5)
plt.grid(True, alpha=0.3)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (m)")
plt.title("Horizontally Shifted Measurements")
#plt.xlim(right=0.5)
# the waves don't all end at the same time
plt.legend(loc='lower left')
plt.show()

In [ ]:
# Add shifted times to trimmed dataframes:
# Note: m3 stays as is because it was used as a reference
M1['t1_hshifted'] = t1_hshifted
M2['t2_hshifted'] = t2_hshifted
M3['t3_hshifted'] = M3["Trimmed Time3 (s)"]
M4['t4_hshifted'] = t4_hshifted
M5['t5_hshifted'] = t5_hshifted
M6['t6_hshifted'] = t6_hshifted

# Plot measurements with shifted times and trim off excess background data:
for i, dframe in enumerate(trimmed_measurements):
    signal_col = f"Detrended/Shifted Measurement {i+1} (m)"
    time_col = f"t{i+1}_hshifted"
    # plot:
    plt.figure(figsize=(12,4))
    plt.plot(dframe[time_col], dframe[signal_col], label=f"Shifted Measurement {i+1}")
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude (m)')
    plt.title(f'Horizontally Shifted Measurement {i+1}')
    #plt.xlim(10,15)
    #plt.legend(loc='upper right')
    plt.grid(True, alpha=0.4)

# Now have square waves starting ~0 for all measurements

#### Align Measurements to Input Signal
- Using cross correlation to align measurement signals (output) to input square wave

In [ ]:
# Trying cross correlation for alignment
def align_measurement_to_input(input_df, input_signal_col, measurement_df, measurement_signal_col, sample_rate=44100):
    """Align a measurement signal to the input signal using cross-correlation.
    Handles different lengths by finding the best matching segment.
    
    input_df: pd.DataFrame with input_signal_col (string) as the signal column
    measurement_df: pd.DataFrame with measurement_signal_col (string) as the signal column
    sample_rate: int, sampling rate in Hz
    
    Returns:
    aligned_signal: np.array, measurement signal aligned and padded to input length
    delay_samples: int, detected delay in samples
    delay_time: float, detected delay in seconds"""
    
    # Extract signal arrays
    input_signal = input_df[input_signal_col].values
    measurement_signal = measurement_df[measurement_signal_col].values
    
    input_length = len(input_signal)
    measurement_length = len(measurement_signal)
    
    # Cross-correlate to find where measurement best matches in input
    correlation = scipy.signal.correlate(input_signal, measurement_signal, mode='full')
    
    # Find the best match index
    best_match_idx = np.argmax(np.abs(correlation))
    
    # Convert to delay in the input signal
    # The correlation array is centered, so need to adjust
    delay_samples = best_match_idx - (measurement_length - 1)
    
    # Create aligned signal by padding/truncating as needed
    aligned_signal = np.zeros(input_length)
    
    # Determine the overlap region
    start_in_input = max(0, delay_samples)
    end_in_input = min(input_length, delay_samples + measurement_length)
    
    start_in_measurement = max(0, -delay_samples)
    end_in_measurement = start_in_measurement + (end_in_input - start_in_input)
    
    # Copy the overlapping portion
    aligned_signal[start_in_input:end_in_input] = measurement_signal[start_in_measurement:end_in_measurement]
    
    delay_time = delay_samples / sample_rate
    
    return aligned_signal, delay_samples, delay_time

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
square_wave_df = pd.DataFrame({"Square Wave": square_wave, "Time in Seconds": t})
# Example usage for all measurements
aligned_measurements = []
delay_samples_list = []
delay_times_list = []

for i, meas_df in enumerate(trimmed_measurements):
    sig_col = f"Detrended/Shifted Measurement {i+1} (m)"
    aligned_sig, delay_samp, delay_time = align_measurement_to_input(square_wave_df, 'Square Wave', meas_df, sig_col, sample_rate=44100)
    
    print(f"Measurement {i+1}: Delay = {delay_time:.6f} seconds ({delay_samp} samples)") 
    
    assert len(aligned_sig) == len(square_wave_df['Square Wave'].values)
    
    aligned_measurements.append(aligned_sig)
    delay_samples_list.append(delay_samp)
    delay_times_list.append(delay_time)
    
print("\nStored delays:")
print(f"delay_samples_list: {delay_samples_list}")
print(f"delay_times_list: {delay_times_list}")

In [ ]:
# Function to create alignment verification plots
def plot_alignment_verification(input_df, input_signal_col, input_time_col, measurement_dfs, aligned_measurements, delays_samples, sample_rate=44100):
    """Create plots showing:
    1. Original measurements vs input (showing the delay)
    2. Aligned measurements vs input (should overlap well)"""
    
    input_signal = input_df[input_signal_col].values
    input_time = input_df[input_time_col].values
    
    n_measurements = len(measurement_dfs)
    
    # Create figure with subplots
    fig, axes = plt.subplots(n_measurements, 2, figsize=(15, 3*n_measurements))
    
    if n_measurements == 1:
        axes = axes.reshape(1, -1)
    
    for i, (meas_df, aligned_sig, delay) in enumerate(zip(measurement_dfs, aligned_measurements, delays_samples)):
        measurement_signal_col = f"Detrended/Shifted Measurement {i+1} (m)"
        measurement_time_col = f"t{i+1}_hshifted"
        meas_signal = meas_df[measurement_signal_col].values
        meas_time = meas_df[measurement_time_col].values

        # Normalize measurement for visualization only
        # meas_norm = meas_signal / np.max(np.abs(meas_signal))
        # aligned_norm = aligned_sig / np.max(np.abs(aligned_sig))
        # Normalize using the ORIGINAL measurement's max for both plots
        meas_max = np.max(np.abs(meas_signal))
        meas_norm = meas_signal / meas_max
        aligned_norm = aligned_sig / meas_max  # Use same normalization!
        
        # Left plot: Original (unaligned)
        axes[i, 0].plot(input_time, input_signal, label='Input (normalized)', alpha=0.7, linewidth=2)
        axes[i, 0].plot(meas_time, meas_norm, label='Measurement (normalized for viz)', alpha=0.7, linewidth=1)
        axes[i, 0].set_xlabel('Time (s)')
        axes[i, 0].set_ylabel('Normalized Amplitude')
        axes[i, 0].set_title(f'Measurement {i+1} - BEFORE Alignment\nDelay: {delay/sample_rate:.3f}s')
        axes[i, 0].legend(loc='upper right')
        axes[i, 0].grid(True, alpha=0.3)
        
        # Right plot: Aligned
        axes[i, 1].plot(input_time, input_signal, label='Input (normalized)', alpha=0.7, linewidth=2)
        axes[i, 1].plot(input_time, aligned_norm, label='Measurement (normalized for viz)', alpha=0.7, linewidth=1)
        axes[i, 1].set_xlabel('Time (s)')
        axes[i, 1].set_ylabel('Normalized Amplitude')
        axes[i, 1].set_title(f'Measurement {i+1} - AFTER Alignment')
        axes[i, 1].legend(loc='upper right')
        axes[i, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

# Create the verification plots ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
fig = plot_alignment_verification(square_wave_df,"Square Wave", "Time in Seconds", trimmed_measurements,  # list of measurement dataframes
    aligned_measurements, delay_samples_list)
plt.show()

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Function to compute and verify correlation quality
def check_alignment_quality(input_signal, aligned_measurements):
    """Compute correlation coefficients to verify alignment quality"""
    print("\nAlignment Quality (Correlation Coefficients):")
    print("-" * 50)
    
    for i, aligned_sig in enumerate(aligned_measurements):
        # Pearson correlation coefficient
        corr = np.corrcoef(input_signal, aligned_sig)[0, 1]
        print(f"Measurement {i+1}: r = {corr:.4f}")
    
    print("\n(Values close to ±1 indicate good alignment)")

# Check alignment quality
input_signal = square_wave_df['Square Wave'].values
check_alignment_quality(input_signal, aligned_measurements)

In [ ]:
def plot_alignment(input_df, input_signal_col, input_time_col, measurement_dfs, aligned_measurements, delays_samples, sample_rate=44100):
    """Create plots showing aligned measurements vs input (should overlap well)"""
    input_signal = input_df[input_signal_col].values
    input_time = input_df[input_time_col].values
    
    n_measurements = len(measurement_dfs)
    
    # Create figure with subplots
    fig, axes = plt.subplots(n_measurements, 1, figsize=(15, 3*n_measurements))
    
    for i, (meas_df, aligned_sig, delay) in enumerate(zip(measurement_dfs, aligned_measurements, delays_samples)):
        measurement_signal_col = f"Detrended/Shifted Measurement {i+1} (m)"
        measurement_time_col = f"t{i+1}_hshifted"
        meas_signal = meas_df[measurement_signal_col].values
        meas_time = meas_df[measurement_time_col].values

        # Normalize measurement for visualization only
        meas_norm = meas_signal / np.max(np.abs(meas_signal))
        aligned_norm = aligned_sig / np.max(np.abs(aligned_sig))
        
        # Right plot: Aligned
        axes[i].plot(input_time, input_signal, label='Input (normalized)', alpha=0.7, linewidth=2)
        axes[i].plot(input_time, aligned_norm, label='Measurement (normalized for viz)', alpha=0.7, linewidth=1)
        axes[i].set_xlabel('Time (s)')
        axes[i].set_ylabel('Normalized Amplitude')
        axes[i].set_title(f'Measurement {i+1} - AFTER Alignment')
        axes[i].legend(loc='upper right')
        axes[i].set_xlim(2,2.25)
        axes[i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

# Create the verification plots ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
fig = plot_alignment(square_wave_df,"Square Wave", "Time in Seconds", trimmed_measurements,  # list of measurement dataframes
    aligned_measurements, delay_samples_list)
plt.show()

**Note:**
- Negative correlation results indicate that the signal is inverted (180° phase shift), the transfer function should show this phase relationship

In [ ]:
# Method 2: Peak detection
# Align reseampled measurements in time
# try to automate peak finding for all measurements:
# for i, dframe in enumerate(measurements):
#     time_col = f"Time{i+1} (s)"
#     signal_col = f"Signal{i+1} (m)"
#     #measurement_array = np.array(dframe[signal_col])
#     #time_array = np.array(dframe[time_col])
#     dframe = dframe[dframe[time_col] >= 1.5]
#     dframe = dframe[dframe[time_col] <= 8]
#     resampled_measurement = signal.resample_poly(dframe[signal_col], 441, 500)
#     resampled_time = np.linspace(dframe[time_col].iloc[0], dframe[time_col].iloc[-1], len(resampled_measurement))
#     # get peaks
#     peaks, _ = signal.find_peaks(resampled_measurement, height = 1.0e-5)
#     peak_x = resampled_time[peaks]
#     peak_y = resampled_measurement[peaks]
#     # Print peak locations and values:
#     print(f"Measurement {i+1}:")
#     print("Index of first peak:", peaks[0])
#     print("Time (s) of first x peak:", peak_x[0])
#     print("Amplitude (m) of first y peak:", peak_y[0])
#     # Plot:
#     plt.figure(figsize=(12, 4))
#     plt.plot(resampled_time, resampled_measurement, label='Original signal')
#     plt.scatter(peak_x, peak_y, color='red', marker='x', s=100, label='Detected peaks')
#     plt.xlabel('Time (s)')
#     plt.ylabel('Amplitude (m)')
#     plt.ticklabel_format(axis='y', style="scientific", scilimits=(0,1))
#     plt.xlim(1.5,5)
#     plt.title(f"M{i+1} Peak Detection")
#     plt.legend(loc='lower left')
#     plt.grid(True)
#     plt.show()

In [ ]:
# turn aligned_measurements list into a dataframe:
# Create a DataFrame with each measurement as a column
aligned_df = pd.DataFrame({
    #'time': square_wave_df["Time in Seconds"].values, #input_df['time'].values,
    #'input': square_wave_df["Square Wave"].values,# input_df['signal'].values,
    'measurement_1': aligned_measurements[0],
    'measurement_2': aligned_measurements[1],
    'measurement_3': aligned_measurements[2],
    'measurement_4': aligned_measurements[3],
    'measurement_5': aligned_measurements[4],
    'measurement_6': aligned_measurements[5]})

#print(aligned_df.head())
print(f"\nShape: {aligned_df.shape}")

***
#### Tapering the Time Series
- Taper the aligned results from using cross correlation
- A Hann window is used for tapering
- The taper is set to start at 10% from the data edges, but this can easily be adjusted when calling the apply_hann_window function

In [ ]:
# updated function copied from vibrometer experiment notebook
def apply_hann_window(data, column, taper_fraction = 0.1):
    """Taper edges of time series with Hann window
    taper_fraction = 0.1 will start taper at 10% from edges"""
    n= len(data)
    taper_length = int(n * taper_fraction)
    
    # Create a copy of the data
    windowed_data = data[column].copy()
    windowed_data = windowed_data.astype(float)
    
    # Apply Hann window to the left edge
    left_window = np.hanning(2 * taper_length)[:taper_length]
    windowed_data.iloc[:taper_length] *= left_window
    
    # Apply Hann window to the right edge
    right_window = np.hanning(2 * taper_length)[taper_length:]
    windowed_data.iloc[-taper_length:] *= right_window
    
    return windowed_data
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Plot tapered input with tapered/aligned measurements
for i, signal_col in enumerate(aligned_df):  # note: all measurements are in 1 dataframe this time
    signal_col = f"measurement_{i+1}"
    time = square_wave_df["Time in Seconds"].values
    
    tapered_data = []
    measurement_hann_result = apply_hann_window(aligned_df, signal_col, taper_fraction = 0.1) # taper all measurements
    # normalize measurements for visualization:
    norm_hann_result = measurement_hann_result/np.max(np.abs(measurement_hann_result))
    tapered_data.append(norm_hann_result)

    tapered_input = apply_hann_window(square_wave_df, "Square Wave", taper_fraction = 0.1)
    
    # Plot:
    plt.figure(figsize=(12,4))
    plt.plot(time, tapered_input, label=f"Tapered Input", alpha=0.6)
    plt.plot(time, tapered_data[-1], label=f"Tapered Measurement")
    plt.xlabel("Time (s)")
    plt.ylabel("Normalized Amplitude")
    plt.grid(True, alpha=0.5)
    plt.title(f"Tapered Data - Measurement {i+1} and Input")
    plt.legend(loc='upper right')
    #plt.xlim(left=6)
    plt.tight_layout()
    plt.show()

##### Trim Signals More Before Tapering:

In [ ]:
def trim_to_signal_content(input_signal, measurement_signal, threshold=0.01):
    """Trim both signals to the region where measurement has actual content.
    Parameters:
    input_signal: array
    measurement_signal: array
    threshold: float, Fraction of max amplitude to detect signal start/end
    Returns:
    trimmed_input, trimmed_measurement, start_idx, end_idx"""
    # Find where measurement signal exceeds threshold
    abs_measurement = np.abs(measurement_signal)
    max_amp = np.max(abs_measurement)
    signal_mask = abs_measurement > (threshold * max_amp)
    
    # Find first and last indices with signal
    signal_indices = np.where(signal_mask)[0]
    
    if len(signal_indices) == 0:
        raise ValueError("No signal detected above threshold")
    
    start_idx = signal_indices[0]
    end_idx = signal_indices[-1] + 1
    
    # Trim both signals to this region
    trimmed_input = input_signal[start_idx:end_idx]
    trimmed_measurement = measurement_signal[start_idx:end_idx]
    
    return trimmed_input, trimmed_measurement, start_idx, end_idx

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Align --> trim --> taper
trimmed_measurement_dfs = []
trimmed_input_dfs = []
for i, dframe in enumerate(trimmed_measurements):
    sig_col= f"Detrended/Shifted Measurement {i+1} (m)"
    aligned_measurement, delay_samples, delay_time = align_measurement_to_input(square_wave_df, 'Square Wave', dframe, sig_col, sample_rate=44100)

    # 2. Trim to actual signal content
    trimmed_input, trimmed_measurement, start_idx, end_idx = trim_to_signal_content(square_wave_df['Square Wave'].values, 
                                                                                    aligned_measurement, threshold=0.1)
    # Create DataFrames for the trimmed signals
    trimmed_input_df = pd.DataFrame({f'Trimmed Input {i+1}': trimmed_input})
    trimmed_input_dfs.append(trimmed_input_df)
    #trimmed_measurement_df = pd.DataFrame({'Trimmed Measurement': trimmed_measurement})
    trimmed_measurement_df = pd.DataFrame({f'Trimmed Measurement {i+1}': trimmed_measurement})
    trimmed_measurement_dfs.append(trimmed_measurement_df)

    print(f"Original length: {len(square_wave_df['Square Wave'])}")
    print(f"M{i+1} Trimmed length: {len(trimmed_input)}")
    print(f"Trimmed region: {start_idx} to {end_idx}")
    print("")
    
    # Apply tapering function
    tapered_input = apply_hann_window(trimmed_input_df, f'Trimmed Input {i+1}', taper_fraction=0.1)
    tapered_measurement = apply_hann_window(trimmed_measurement_df, f'Trimmed Measurement {i+1}', taper_fraction=0.1)
    
    # make normalized version for plotting:
    tapered_measurement_norm = tapered_measurement.values / np.max(np.abs(tapered_measurement.values))

    # get time for plotting:
    #trimmed_time = square_wave_df["Time in Seconds"].values[start_idx:end_idx]
    trimmed_time = np.arange(len(trimmed_input)) / 44100
    
    # Plot:
    plt.figure(figsize=(12,4))
    plt.plot(trimmed_time, tapered_input, label=f"Tapered Input", alpha=0.6)
    plt.plot(trimmed_time, tapered_measurement_norm, label=f"Tapered Measurement")
    plt.xlabel("Time (s)")
    plt.ylabel("Normalized Amplitude")
    plt.grid(True, alpha=0.5)
    plt.title(f"Tapered Data - Measurement {i+1} and Input")
    plt.legend(loc='upper right')
    #plt.xlim(left=4)
    #plt.xlim(0,0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot tapered input signal
# from wave generation cell: square_wave = signal.square(2 * np.pi * freq * t)
                            # t = np.linspace(0, duration, int(fs * duration))
square_wave_df = pd.DataFrame({"Square Wave": square_wave, "Time in Seconds": t})
windowed_input = apply_hann_window(square_wave_df, "Square Wave", taper_fraction=0.1)
# Plot:
plt.figure(figsize=(12,4))
plt.plot(square_wave_df["Time in Seconds"], square_wave_df["Square Wave"], label='Input Square Wave', alpha=0.5)
plt.plot(square_wave_df["Time in Seconds"], windowed_input, label="Tapered Input Square Wave")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (m)")
plt.legend(loc='upper right')
plt.title("Input Square Wave - Full and Tapered")
plt.grid(True, alpha=0.5)
plt.show()

In [ ]:
# put tapered measurement data (not normalized) into dataframe:
# tapered_measurements = [] # Initialize list BEFORE loop
# for i, signal_col in enumerate(aligned_df):
#     signal_col = f"measurement_{i+1}"
#     measurement_hann_result = apply_hann_window(aligned_df, signal_col, taper_fraction = 0.2) # taper all measurements, don't normalize
#     tapered_measurements.append(measurement_hann_result)

# tapered_measurements_df = pd.DataFrame({
#     'tapered_m1': tapered_measurements[0],
#     'tapered_m2': tapered_measurements[1],
#     'tapered_m3': tapered_measurements[2],
#     'tapered_m4': tapered_measurements[3],
#     'tapered_m5': tapered_measurements[4],
#     'tapered_m6': tapered_measurements[5]})

# print(tapered_measurements_df.head())
# print(f"\nShape: {tapered_measurements_df.shape}")

***
#### Fast Fourier Transform
- Plotting fast Fourier transform (FFT) result for each output measurement with FFT result from input signal

In [ ]:
# coherent gain function copied from vibrometer experiment notebook
# def calculate_coherent_gain(data, taper_fraction=0.1):
#     """Calculate coherent gain for partial Hann taper"""
#     n=len(data)
#     taper_length = int(n * taper_fraction)
    
#     # Create the full window
#     window = np.ones(n)
    
#     # Left taper
#     left_window = np.hanning(2 * taper_length)[:taper_length]
#     window[:taper_length] = left_window
    
#     # Right taper
#     right_window = np.hanning(2 * taper_length)[taper_length:]
#     window[-taper_length:] = right_window
    
#     # Coherent gain is the mean
#     coherent_gain = np.mean(window)
    
#     return coherent_gain

In [ ]:
# OLD VERSION:
# def plot_fft(data, column, sr, fig1=None, fig2=None, fig3=None, name='', apply_window=True):
#     """Plot FFT with windowing correction.
#     data: array
#     column: relevant data column from data
#     sr: Sampling rate
#     apply_window: bool, says whether to apply Hann window (default True)"""
#     n = len(data)
#     taper_fraction = 0.1 # taper will start at 10% from data edges (change as needed), if apply_window=True
    
#     # Apply window if requested
#     if apply_window:
#         windowed_data = apply_hann_window(data, column, taper_fraction= taper_fraction)
        
#         # Calculate coherent gain for partial taper
#         taper_length = int(n * taper_fraction)
#         window = np.ones(n)
#         left_window = np.hanning(2 * taper_length)[:taper_length]
#         window[:taper_length] = left_window
#         right_window = np.hanning(2 * taper_length)[taper_length:]
#         window[-taper_length:] = right_window
#         coherent_gain = np.mean(window)
        
#     else:
#         windowed_data = data[column].values
#         coherent_gain = 1.0

#     # Compute FFT
#     fft_values = scipy.fft.rfft(windowed_data)
#     freqs = scipy.fft.rfftfreq(n, d=1/sr)
    
#     # magnitudes WITH coherent gain correction
#     magnitudes = np.abs(fft_values) / (n * coherent_gain)
    
#     # Double the AC components (not DC and Nyquist)
#     magnitudes[1:] *= 2.0
    
#     # Undo doubling for Nyquist if even length
#     if n % 2 == 0:
#         magnitudes[-1] /= 2.0
    
#     phases = np.angle(fft_values)
    
#     # Create figures:
#     if fig1 is None:
#         fig1 = plt.figure(figsize=(10,6))
#         plt.plot(freqs, magnitudes, linewidth=1, label=name)
#         plt.title('FFT Magnitude Spectrum (Linear)', fontsize=16)
#         plt.xlabel('Frequency (Hz)', fontsize=14)
#         plt.ylabel('Magnitude', fontsize=14)
#         plt.grid(True, which='major', alpha=0.5)
#         plt.grid(True, which='minor', alpha=0.5)
#         plt.minorticks_on()
#         plt.xticks(fontsize=14)
#         plt.yticks(fontsize=14)
#         #plt.xlim(0,200)
#         plt.tight_layout()
#         if name:
#             plt.legend(loc='upper right')
#     else:
#         plt.figure(fig1.number)
#         plt.plot(freqs, magnitudes, linewidth=1, label=name, alpha=0.5)
#         if name:
#             plt.legend(loc='upper right')
    
#     if fig2 is None:
#         fig2 = plt.figure(figsize=(10, 6))
#         plt.loglog(freqs[1:], magnitudes[1:], linewidth=1, label=name)
#         plt.title('FFT Magnitude Spectrum (log)', fontsize=16)
#         plt.xlabel('Frequency (Hz)', fontsize=14)
#         plt.ylabel('Magnitude', fontsize=14)
#         plt.xticks(fontsize=14)
#         plt.yticks(fontsize=14)
#         plt.grid(True, which='major', alpha=0.5)
#         plt.grid(True, which='minor', alpha=0.5)
#         #plt.xlim(10,1000)
#         plt.tight_layout()
#         if name:
#             plt.legend(loc='upper right')
#     else:
#         plt.figure(fig2.number)
#         plt.loglog(freqs[1:], magnitudes[1:], linewidth=1, label=name, alpha=0.5)
#         plt.legend()
#         if name:
#             plt.legend(loc='upper right')

#     if fig3 is None:
#         fig3 = plt.figure(figsize=(10, 6))
#         plt.plot(freqs[1:], phases[1:], linewidth=1, label=name)
#         plt.title('Phase Spectrum', fontsize=16)
#         plt.xlabel('Frequency (Hz)', fontsize=14)
#         plt.ylabel('Phase (radians)', fontsize=14)
#         plt.grid(True, alpha=0.5)
#         plt.xticks(fontsize=14)
#         plt.yticks(fontsize=14)
#         plt.xlim(-1, 50)
#         plt.tight_layout()
#         if name:
#             plt.legend(loc='lower right', framealpha=0.5)
#     else:
#         plt.figure(fig3.number)
#         plt.plot(freqs[1:], phases[1:], linewidth=1, label=name)
#         if name:
#             plt.legend(loc='lower right', framealpha=0.8)
    
#     return fig1, fig2, fig3 
    
# #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# # for i, dframe in enumerate(resampled_measurements):
# #     signal_col = f"Detrended/Shifted Measurement {i+1} (m)"
# #     time_col = f"t{i+1}_resamp_shifted"
# #     dframe = dframe[dframe[time_col] >= 1.5]
# #     dframe = dframe[dframe[time_col] <= 6.4]
# #     # For measurement data:
# #     fig1, fig2, fig3 = plot_fft(dframe, signal_col, sr=44100, name=f'Output Measurement {i+1}', apply_window=True)
# #     # For input signal:
# #     plot_fft(square_wave_df, "Square Wave", sr=44100, fig1=fig1, fig2=fig2, fig3=fig3, name='Input Signal', apply_window=True)
# #     plt.show()
# #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# for i, (input, output) in enumerate(zip(trimmed_input_dfs, trimmed_measurement_dfs)):
#     output_signal_col = f"Trimmed Measurement {i+1}"
#     input_signal_col = f"Trimmed Input {i+1}"
#     # For measurement data:
#     fig1, fig2, fig3 = plot_fft(output, output_signal_col, sr=44100, name=f"Measurement {i+1}", apply_window=True)
#     # For input signal:
#     plot_fft(input, input_signal_col, sr=44100, fig1=fig1, fig2=fig2, fig3=fig3, name="Input Signal", apply_window=True)
#     plt.show()

##### FFT With Frequency Bin Averaging

In [ ]:
def plot_averaged_fft(data, column, sr, segment_size, title=''):
    """Plot FFT with frequency bin averaging 
    data: pandas DataFrame
    column: relevant data column from data, string
    sr: Sampling rate in Hz
    segment_size:segment size for averaging, int
    title: plot title, str"""
    # Goal: decrease frequency resolution of the FFT
    # Signal parameters
    T = 1 / sr  # Sampling period
    # signal x:
    x = data[column].values
    N = len(x)
    
    # Parameters for averaging
    overlap = 0.5   # 50% overlap
    step_size = int(segment_size * (1 - overlap))
    
    # Calculate number of segments
    num_segments = (N - segment_size) // step_size + 1
    
    # Initialize array to store summed FFT results
    avg_fft_magnitude = np.zeros(segment_size // 2 + 1) # only need one side for real input
    
    for i in range(num_segments):
        start = i * step_size
        end = start + segment_size
        segment = x[start:end]
        
        # apply a Hanning window to the segment to reduce spectral leakage
        window = np.hanning(segment_size)
        windowed_segment = segment * window
        
        # Compute FFT of the segment
        segment_fft = scipy.fft.rfft(windowed_segment, segment_size)
        
        # Get the magnitude of the positive frequency side
        #segment_magnitude = np.abs(segment_fft)[:segment_size // 2 + 1]
        segment_magnitude = np.abs(segment_fft)[:segment_size // 2 + 1] / np.sum(window) # window sum is for amplitude correction
        
        # Accumulate the magnitudes
        #avg_fft_magnitude += segment_magnitude 
        avg_fft_magnitude += segment_magnitude ** 2 #accumulate power
    
    # Calculate the mean magnitude
    #avg_fft_magnitude /= num_segments
    avg_fft_magnitude = np.sqrt(avg_fft_magnitude / num_segments) # don't take square root if want power spectral density instead
    
    # Get the frequency bins
    freqs = scipy.fft.rfftfreq(segment_size, T)[:segment_size // 2 + 1]
    
    # Plot the averaged frequency spectrum
    plt.figure(figsize=(12, 6))
    #plt.plot(freqs, avg_fft_magnitude)
    plt.loglog(freqs, avg_fft_magnitude)
    plt.xlabel('Frequency (Hz)', fontsize=12)
    plt.ylabel('Averaged FFT Magnitude', fontsize=12)
    plt.title(title)
    plt.grid(True, which='both', alpha=0.5)
    plt.show()

    return freqs, avg_fft_magnitude
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
for i, (input, output) in enumerate(zip(trimmed_input_dfs, trimmed_measurement_dfs)):
    output_signal_col = f"Trimmed Measurement {i+1}"
    input_signal_col = f"Trimmed Input {i+1}"
    title_out = f"Averaged Frequency Spectrum for Output {i+1}"
    title_in = f"Averaged Frequency Spectrum for Input {i+1}"
    freqs_out, avg_fft_out = plot_averaged_fft(output, output_signal_col, 44100, segment_size=1000, title=title_out) 
    freqs_in, avg_fft_in = plot_averaged_fft(input, input_signal_col, 44100, segment_size=1000, title=title_in)

##### Power Spectral Density With Welch's Method:
- Change the scaling parameter in welch() from 'density' to 'spectrum' to plot a squared magnitude spectrum instead

In [ ]:
def welch_psd(data, column, sr, title, segment_size, window='hann'):
    """data: pandas DataFrame
    column: relevant data column from data, str
    sr: Sampling rate in Hz
    title: title for PSD plot, str
    segment_size: length of each segment
    window: desired window to use"""
    
    overlap = segment_size//2 # number of points to overlap between segments, here is 50% overlap
    freqs, psd = welch(data[column].values, fs=sr, nperseg=segment_size, noverlap=overlap, window=window, scaling='density')

    # Plot:
    plt.figure(figsize=(12,6))
    #plt.plot(freqs,psd)
    plt.loglog(freqs, psd)
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("PSD")
    plt.title(title)
    plt.grid(True, which='both', alpha=0.5)
    #plt.ylim(bottom=10**-4)
    plt.show()

    return freqs, psd
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
for i, (input, output) in enumerate(zip(trimmed_input_dfs, trimmed_measurement_dfs)):
    output_signal_col = f"Trimmed Measurement {i+1}"
    input_signal_col = f"Trimmed Input {i+1}"
    title_out = f"Averaged Frequency Spectrum for Output {i+1}"
    title_in = f"Averaged Frequency Spectrum for Input {i+1}"
    freqs_out_welch, out_psd = welch_psd(output, output_signal_col, 44100, title=title_out, segment_size=1000) 
    freqs_in_welch, in_psd = welch_psd(input, input_signal_col, 44100, title=title_in, segment_size=1000)

***
#### Estimating Transfer Function/Frequency Response

In [ ]:
# VERSION 1 - based on original FFT code
# estimating transfer function using function based on plot_fft above
# def compute_and_plot_transfer_function(input_data, input_column, output_data, output_column, sr, fig1=None, fig2=None, fig3=None, 
#                                        name='', apply_window=True, taper_fraction=0.1):
#     """Compute transfer function from input and output signals: H(f) = FFT(output) / FFT(input)
#     input_data: DataFrame containing input signal
#     input_column: column name for input signal
#     output_data: DataFrame containing output signal  
#     output_column: column name for output signal
#     sr: Sampling rate
#     fig1, fig2, fig3: Optional figure objects for subplots
#     name: Label for plots
#     apply_window: bool, whether to apply Hann window (default True)
#     taper_fraction: fraction for tapering
    
#     Returns:
#     fig1: Magnitude spectrum plot (linear)
#     fig2: Magnitude spectrum plot (log)
#     fig3: Phase spectrum plot
#     freqs: Frequency array
#     H: transfer function"""
#     n_input = len(input_data)
#     n_output = len(output_data)
    
#     # Ensure both signals are same length
#     if n_input != n_output:
#         print(f"Warning: Input length ({n_input}) != Output length ({n_output})")
#         n = min(n_input, n_output)
#     else:
#         n = n_input
    
#     # Trim BEFORE windowing to ensure all arrays are consistent length
#     input_data  = input_data.iloc[:n]
#     output_data = output_data.iloc[:n]
    
#     # Apply window if requested
#     if apply_window:
#         input_windowed  = apply_hann_window(input_data,  input_column,  taper_fraction=taper_fraction)
#         output_windowed = apply_hann_window(output_data, output_column, taper_fraction=taper_fraction)
    
#         # Calculate coherent gain for reference (not applied to ratio)
#         taper_length = int(n * taper_fraction)
#         window = np.ones(n)
#         left_window  = np.hanning(2 * taper_length)[:taper_length]
#         right_window = np.hanning(2 * taper_length)[taper_length:]
#         window[:taper_length]  = left_window
#         window[-taper_length:] = right_window
#         coherent_gain = np.mean(window)  # informational only
    
#     else:
#         input_windowed  = input_data[input_column].values
#         output_windowed = output_data[output_column].values
#         coherent_gain   = 1.0
    
#     # Compute FFTs
#     input_fft = scipy.fft.rfft(input_windowed)
#     output_fft = scipy.fft.rfft(output_windowed)
#     freqs = scipy.fft.rfftfreq(n, d=1/sr)
    
#     # Compute transfer function H(f) = Output(f) / Input(f)
#     # Add small epsilon to avoid division by zero?
#     #epsilon = 1e-10
#     epsilon = 1e-10 * np.max(np.abs(input_fft))  # scale-relative epsilon?
#     H = output_fft / (input_fft + epsilon)

#     # Compute magnitude and phase
#     magnitudes = np.abs(H) # don't divide be coherent_gain, the window effect cancels in the ratio
#     phases = np.angle(H, deg=True) 

#     # Note: The magnitude doubling cancels in H(f) = |FFT_out| / |FFT_in| since both sides
#     # are real signals and would be doubled equally.
    
#     # Convert magnitude to dB?
#     #magnitudes_db = 20 * np.log10(magnitudes + epsilon)
    
#     # Create figures:
#     # Figure 1: Linear magnitude
#     if fig1 is None:
#         fig1 = plt.figure(figsize=(10, 6))
#         plt.plot(freqs, magnitudes, linewidth=1, label=name)
#         plt.title('Transfer Function Magnitude (Linear)', fontsize=16)
#         plt.xlabel('Frequency (Hz)', fontsize=14)
#         plt.ylabel('Magnitude |H(f)|', fontsize=14)
#         plt.grid(True, which='major', alpha=0.5)
#         plt.grid(True, which='minor', alpha=0.5)
#         plt.minorticks_on()
#         plt.xticks(fontsize=14)
#         plt.yticks(fontsize=14)
#         plt.tight_layout()
#         if name:
#             plt.legend(loc='upper right')
#     else:
#         plt.figure(fig1.number)
#         plt.plot(freqs, magnitudes, linewidth=1, label=name, alpha=0.5)
#         if name:
#             plt.legend(loc='upper right')
    
#     # Figure 2: Log magnitude
#     if fig2 is None:
#         fig2 = plt.figure(figsize=(10, 6))
#         plt.loglog(freqs[1:], magnitudes[1:], linewidth=1, label=name)
#         plt.title('Transfer Function Magnitude (log)', fontsize=16)
#         plt.xlabel('Frequency (Hz)', fontsize=14)
#         plt.ylabel('Magnitude |H(f)| ', fontsize=14)
#         plt.xticks(fontsize=14)
#         plt.yticks(fontsize=14)
#         plt.grid(True, which='major', alpha=0.5)
#         plt.grid(True, which='minor', alpha=0.5)
#         plt.tight_layout()
#         if name:
#             plt.legend(loc='upper right')
#     else:
#         plt.figure(fig2.number)
#         plt.loglog(freqs[1:], magnitudes[1:], linewidth=1, label=name, alpha=0.5)
#         plt.legend()
#         if name:
#             plt.legend(loc='upper right')
    
#     # Figure 3: Phase
#     if fig3 is None:
#         fig3 = plt.figure(figsize=(10, 6))
#         plt.plot(freqs[1:], phases[1:], linewidth=1, label=name)
#         plt.title('Transfer Function Phase', fontsize=16)
#         plt.xlabel('Frequency (Hz)', fontsize=14)
#         plt.ylabel('Phase (degrees)', fontsize=14)
#         plt.grid(True, alpha=0.5)
#         plt.xticks(fontsize=14)
#         plt.yticks(fontsize=14)
#         plt.xlim(-1,50) # zoom in to see details
#         plt.tight_layout()
#         if name:
#             plt.legend(loc='upper left', framealpha=0.5)
#     else:
#         plt.figure(fig3.number)
#         plt.plot(freqs[1:], phases[1:], linewidth=1, label=name, alpha=0.8)
#         if name:
#             plt.legend(loc='upper left', framealpha=0.8)
    
#     return fig1, fig2, fig3, freqs, H, magnitudes, phases

# #Compute transfer functions for all measurements ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# for i, (input, output) in enumerate(zip(trimmed_input_dfs, trimmed_measurement_dfs)):
#     output_signal_col = f"Trimmed Measurement {i+1}"
#     input_signal_col = f"Trimmed Input {i+1}"
    
#     # Compute transfer function
#     fig1, fig2, fig3, freqs, H, mags, phases = compute_and_plot_transfer_function(input_data=input, input_column=input_signal_col,
#                                                 output_data=output, output_column=output_signal_col, sr=44100, fig1=None, 
#                                                 fig2=None, fig3=None, name=f"Measurement {i+1}", apply_window=True)

# plt.show()

##### Estimating Transfer Function Using Welch's Method:
- Note: The H1 estimator minimizes the effect of output noise

In [ ]:
# VERSION 2
# ADDING WELCH/PSD AS ALTERNATIVE METHOD HERE:
from scipy.signal import csd, welch, coherence

def welch_transfer_function(input_data, input_column, output_data, output_column, sr, title, segment_size, window='hann'):
    """Estimate transfer function using Welch's method.
    input_data/output_data: pandas DataFrames
    input_column/output_column: relevant column names, str
    sr: sampling rate in Hz
    title: title for plots, str
    segment_size: length of each segment (nperseg), controls frequency resolution
    window: window function to use
    
    Returns: freqs, H, magnitudes, phases_deg, coherence"""
    overlap = segment_size // 2
    x = input_data[input_column].values
    y = output_data[output_column].values

    # power spectral density of input
    freqs, Pxx = welch(x, fs=sr, nperseg=segment_size, noverlap=overlap, window=window, scaling='density')

    # Cross-power spectral density (output vs input)
    freqs, Pyx = csd(x, y, fs=sr, nperseg=segment_size, noverlap=overlap, window=window, scaling='density')

    # H1 estimator: H(f) = Pyx/Pxx
    H = Pyx/Pxx
    magnitudes = np.abs(H)
    phases_deg = np.degrees(np.angle(H))

    # Coherence: quality metric, close to 1.0 indicates good estimate
    freqs_coh, Cxy = coherence(x, y, fs=sr, nperseg=segment_size, noverlap=overlap, window=window)

    # Plots
    fig, axes = plt.subplots(3, 1, figsize=(12, 12))

    axes[0].loglog(freqs[1:], magnitudes[1:]) # non-dc = freqs > 0
    axes[0].set_xlabel("Frequency (Hz)")
    axes[0].set_ylabel("Magnitude |H(f)|")
    axes[0].set_title(f"{title} - Transfer Function Magnitude")
    axes[0].grid(True, which='both', alpha=0.5)

    #axes[1].semilogx(freqs[1:], phases_deg[1:])
    axes[1].plot(freqs, phases_deg)
    axes[1].set_xlabel("Frequency (Hz)")
    axes[1].set_ylabel("Phase (degrees)")
    axes[1].set_title(f"{title} - Transfer Function Phase")
    axes[1].grid(True, which='both', alpha=0.5)
    
    # Add plot for coherence?
    axes[2].semilogx(freqs_coh[1:], Cxy[1:])
    axes[2].set_xlabel("Frequency (Hz)")
    axes[2].set_ylabel("Coherence")
    axes[2].set_title(f"{title} - Coherence (1.0 = fully linear)")
    axes[2].set_ylim(0, 1.1)
    axes[2].axhline(0.9, color='r', linestyle='--', alpha=0.5, label='0.9 threshold')
    axes[2].legend()
    axes[2].grid(True, which='both', alpha=0.5)

    plt.tight_layout()
    plt.show()

    return freqs, H, magnitudes, phases_deg, Cxy
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
#Test:
for i, (input, output) in enumerate(zip(trimmed_input_dfs, trimmed_measurement_dfs)):
    output_signal_col = f"Trimmed Measurement {i+1}"
    input_signal_col = f"Trimmed Input {i+1}"
    title_ = f"M{i+1}"

    freqs, H, magnitudes, phases_deg, Cxy = welch_transfer_function(input, input_signal_col, output, output_signal_col, 
                                                                    sr=44100, title=title_, segment_size=4096, window='hann')

In [ ]:
# version with bandpass filter:
def welch_transfer_function(input_data, input_column, output_data, output_column, sr, title, segment_size, lowcut, highcut, window='hann'):
    """Estimate transfer function using Welch's method.
    input_data/output_data: pandas DataFrames
    input_column/output_column: relevant column names, str
    sr: sampling rate in Hz
    title: title for plots, str
    segment_size: length of each segment (nperseg), controls frequency resolution
    lowcut: lowest frequency for bandpass filter, int
    highcut: highest frequency for bandpass filter, int
    window: window function to use
    
    Returns: freqs, H, magnitudes, phases_deg, coherence"""
    overlap = segment_size // 2
    x = input_data[input_column].values
    y = output_data[output_column].values

    # New change: Apply bandpass filter~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    def bandpass(signal, low, high, fs, order=4):
        # low = lowcut value, high = highcut value
        sos = butter(order, [low, high], btype='band', fs=fs, output='sos')
        return sosfilt(sos, signal)

    x = bandpass(x, low=lowcut, high=highcut, fs=sr) # for 100 Hz square wave
    y = bandpass(y, low=lowcut, high=highcut, fs=sr)

    # power spectral density of input~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    freqs, Pxx = welch(x, fs=sr, nperseg=segment_size, noverlap=overlap, window=window, scaling='density')

    # Cross-power spectral density (output vs input)
    #freqs, Pyx = csd(y, x, fs=sr, nperseg=segment_size, noverlap=overlap, window=window, scaling='density')
    freqs, Pyx = csd(x, y, fs=sr, nperseg=segment_size, noverlap=overlap, window=window, scaling='density')

    # H1 estimator: H(f) = Pyx/Pxx
    H = Pyx/Pxx
    magnitudes = np.abs(H)
    phases_deg = np.degrees(np.angle(H))

    # Coherence: quality metric, close to 1.0 indicates good estimate
    freqs_coh, Cxy = coherence(x, y, fs=sr, nperseg=segment_size, noverlap=overlap, window=window)

    ###~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Apply mask to BOTH coherence arrays together
    coh_mask = (freqs_coh >= lowcut) & (freqs_coh <= highcut) # should be same values used with bandpass function
    freqs_coh = freqs_coh[coh_mask]
    Cxy = Cxy[coh_mask]
    
    # And for the main spectrum:
    band_mask = (freqs >= lowcut) & (freqs <= highcut) # should be same values used with bandpass function
    freqs = freqs[band_mask]
    magnitudes = magnitudes[band_mask]
    phases_deg = phases_deg[band_mask]
    H = H[band_mask]

    # Plots
    fig, axes = plt.subplots(3, 1, figsize=(12, 12))

    axes[0].loglog(freqs, magnitudes) 
    axes[0].set_xlabel("Frequency (Hz)")
    axes[0].set_ylabel("Magnitude |H(f)|")
    axes[0].set_title(f"{title} - Transfer Function Magnitude")
    axes[0].grid(True, which='both', alpha=0.5)

    #axes[1].semilogx(freqs, phases_deg)
    axes[1].plot(freqs, phases_deg)
    axes[1].set_xlabel("Frequency (Hz)")
    axes[1].set_ylabel("Phase (degrees)")
    axes[1].set_title(f"{title} - Transfer Function Phase")
    axes[1].grid(True, which='both', alpha=0.5)
    
    # Add plot for coherence?
    axes[2].semilogx(freqs_coh, Cxy)
    axes[2].set_xlabel("Frequency (Hz)")
    axes[2].set_ylabel("Coherence")
    axes[2].set_title(f"{title} - Coherence")
    axes[2].set_ylim(0, 1.1)
    axes[2].axhline(0.9, color='r', linestyle='--', alpha=0.5, label='0.9 threshold')
    axes[2].legend()
    axes[2].grid(True, which='both', alpha=0.5)

    plt.tight_layout()
    plt.show()

    return freqs, H, magnitudes, phases_deg, Cxy
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
for i, (input, output) in enumerate(zip(trimmed_input_dfs, trimmed_measurement_dfs)):
    output_signal_col = f"Trimmed Measurement {i+1}"
    input_signal_col = f"Trimmed Input {i+1}"
    title_ = f"M{i+1}"

    freqs, H, magnitudes, phases_deg, Cxy = welch_transfer_function(input, input_signal_col, output, output_signal_col, 
                                                                    sr=44100, title=title_, segment_size=9096, lowcut=80, highcut=520, window='hann')